# Task 2: Pop Melody Harmonization with Seq2Seq LSTM

## What is Harmonization?
Given a **melody**, harmonization generates an **accompaniment** that supports it musically.
We take the MELODY track from a POP909 song and generate a PIANO accompaniment.

## Differences from Task 1

| Aspect | Task 1 (Bach Chorales) | Task 2 (POP909) |
|--------|------------------------|-----------------|
| Genre | Baroque (1650–1750) | Modern pop |
| Dataset | ~370 chorales | 909 songs, 70 k+ windows |
| Model | Decoder-only causal Transformer | Seq2Seq BiLSTM + Attention |
| Task | Unconditioned generation | Melody-conditioned harmonization |
| Melody access | Causal (no look-ahead) | Bidirectional (full melody visible) |

The key architectural difference: a harmonizer *should* see the whole melody before
deciding what chords to play. A causal decoder-only transformer can't do that —
a bidirectional encoder can.


## 2. Exploratory Data Analysis

We scan all 909 POP909 songs directly from their MIDI files.

In [ ]:
import sys, os, json, tempfile, subprocess
sys.path.insert(0, 'modeling_task2')

import pretty_midi
import soundfile as sf_audio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from IPython.display import HTML, display
import base64

DATA_DIR   = Path('data/POP909/POP909')
SOUNDFONT  = 'modeling/checkpoints/MuseScore_General.sf3'
SCAN_CACHE = Path('EDA_task2/scan_cache.json')
FLUIDSYNTH = '/opt/homebrew/bin/fluidsynth'  # full path so Jupyter can find it

songs = sorted(DATA_DIR.iterdir())
print(f"Songs found:  {len(songs)}")
print(f"Soundfont:    {SOUNDFONT}  (exists={os.path.exists(SOUNDFONT)})")
print(f"Fluidsynth:   {FLUIDSYNTH}  (exists={os.path.exists(FLUIDSYNTH)})")
print(f"Scan cache:   {SCAN_CACHE}  (exists={SCAN_CACHE.exists()})")

def midi_to_wav(pm_or_path, wav_path):
    """Render MIDI to WAV via fluidsynth subprocess — no audio playback."""
    if isinstance(pm_or_path, pretty_midi.PrettyMIDI):
        with tempfile.NamedTemporaryFile(suffix='.mid', delete=False) as f:
            pm_or_path.write(f.name)
            mid_path = f.name
    else:
        mid_path = str(pm_or_path)
    subprocess.run(
        [FLUIDSYNTH, '-ni', SOUNDFONT, mid_path, '-F', wav_path, '-r', '44100'],
        capture_output=True, timeout=120)
    if os.path.exists(wav_path) and os.path.getsize(wav_path) > 1000:
        print(f"  → {wav_path}  ({os.path.getsize(wav_path)//1024} KB)")
        return True
    print(f"  ✗ WAV not created for {mid_path}")
    return False

def audio_tag(wav_path, title):
    data = base64.b64encode(open(wav_path, 'rb').read()).decode()
    return (f'<div style="margin:10px 0">'
            f'<p><b>{title}</b></p>'
            f'<audio controls style="width:100%">'
            f'<source src="data:audio/wav;base64,{data}" type="audio/wav">'
            f'</audio></div>')


### 2.1 Inspect One Song

Look at the raw MIDI tracks and chord annotations for song 001.

In [ ]:
song_dir = DATA_DIR / '001'
pm = pretty_midi.PrettyMIDI(str(song_dir / '001.mid'))

print(f"Duration: {pm.get_end_time():.1f}s")
print(f"Estimated tempo: {pm.estimate_tempo():.1f} BPM")
print(f"\nTracks ({len(pm.instruments)}):")
for i, inst in enumerate(pm.instruments):
    pitches = [n.pitch for n in inst.notes]
    durs    = [n.end - n.start for n in inst.notes]
    print(f"  [{i}] program={inst.program:3d}  notes={len(inst.notes):4d}"
          f"  pitch=[{min(pitches)},{max(pitches)}]"
          f"  avg_dur={np.mean(durs):.2f}s")

print("\nFirst 8 chord annotations:")
with open(song_dir / 'chord_midi.txt') as f:
    for line in list(f)[:8]:
        parts = line.strip().split('\t')
        print(f"  {float(parts[0]):6.2f}s – {float(parts[1]):6.2f}s  {parts[2]}")


In [ ]:
# Song 001 preview — melody only, piano only, full mix (first 30s)
song_dir = DATA_DIR / '001'
pm_full  = pretty_midi.PrettyMIDI(str(song_dir / '001.mid'))

def trim_pm(pm, end_time=30.0):
    out = pretty_midi.PrettyMIDI(initial_tempo=pm.get_tempo_changes()[1][0])
    for inst in pm.instruments:
        ni = pretty_midi.Instrument(program=inst.program, name=inst.name)
        ni.notes = [n for n in inst.notes if n.start < end_time]
        out.instruments.append(ni)
    return out

def single_track_pm(pm, track_idx, end_time=30.0):
    out  = pretty_midi.PrettyMIDI(initial_tempo=pm.get_tempo_changes()[1][0])
    inst = pm.instruments[track_idx]
    ni   = pretty_midi.Instrument(program=inst.program, name=inst.name)
    ni.notes = [n for n in inst.notes if n.start < end_time]
    out.instruments.append(ni)
    return out

os.makedirs('evaluation_task2', exist_ok=True)
samples = [
    (trim_pm(pm_full, 30),        'evaluation_task2/preview_001_full.wav',   'Song 001 — Full mix (first 30s)'),
    (single_track_pm(pm_full, 0), 'evaluation_task2/preview_001_melody.wav', 'Song 001 — Melody only'),
    (single_track_pm(pm_full, 2), 'evaluation_task2/preview_001_piano.wav',  'Song 001 — Piano/accompaniment only'),
]

print("Rendering previews (silent — no system audio)...")
html = ''
for pm_obj, wav_path, title in samples:
    ok = midi_to_wav(pm_obj, wav_path)
    if ok:
        html += audio_tag(wav_path, title)
    else:
        html += f'<p style="color:red">{title} — render failed</p>'

display(HTML(html))


### 2.2 Melody Pitch Distribution

Scan the MELODY track (track 0) across all 909 songs.

In [ ]:
if SCAN_CACHE.exists():
    print(f"Loading scan cache from {SCAN_CACHE} ...")
    cache = json.load(open(SCAN_CACHE))
    all_pitches    = cache['all_pitches']
    song_lengths   = cache['song_lengths']
    note_densities = cache['note_densities']
    chord_counter  = Counter(cache['chord_counts'])
    print(f"Loaded. ({len(song_lengths)} songs)")
else:
    print("Scanning all 909 songs (first run only — will cache after)...")
    all_pitches, song_lengths, note_densities = [], [], []
    chord_counter = Counter()
    failed = 0
    for i, song_dir in enumerate(sorted(DATA_DIR.iterdir()), 1):
        mid_files = list(song_dir.glob('*.mid'))
        if not mid_files:
            continue
        try:
            pm        = pretty_midi.PrettyMIDI(str(mid_files[0]))
            mel_track = pm.instruments[0]
            duration  = pm.get_end_time()
            notes     = mel_track.notes
            if not notes or duration < 5:
                continue
            all_pitches.extend(n.pitch for n in notes)
            song_lengths.append(duration)
            note_densities.append(len(notes) / duration)
            chord_file = song_dir / 'chord_midi.txt'
            if chord_file.exists():
                with open(chord_file) as f:
                    for line in f:
                        parts = line.strip().split('\t')
                        if len(parts) == 3 and parts[2] != 'N':
                            chord_counter[parts[2]] += 1
        except Exception:
            failed += 1
        if i % 100 == 0:
            print(f"  {i}/909 ...")

    print(f"Done. {len(song_lengths)} songs parsed, {failed} failed.")
    SCAN_CACHE.parent.mkdir(exist_ok=True)
    json.dump({'all_pitches': all_pitches, 'song_lengths': song_lengths,
               'note_densities': note_densities, 'chord_counts': dict(chord_counter)},
              open(SCAN_CACHE, 'w'))
    print(f"Cached to {SCAN_CACHE}")

print(f"\nTotal melody notes: {len(all_pitches):,}")
print(f"Pitch range:        {min(all_pitches)} – {max(all_pitches)}")
print(f"Mean pitch:         {np.mean(all_pitches):.1f}  (std {np.std(all_pitches):.1f})")
print(f"Avg song length:    {np.mean(song_lengths):.1f}s")
print(f"Avg note density:   {np.mean(note_densities):.2f} notes/s")
print(f"Unique chord types: {len(chord_counter)}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(all_pitches, bins=60, color='steelblue', edgecolor='white', lw=0.3)
axes[0].axvline(np.mean(all_pitches), color='red', ls='--', label=f'Mean: {np.mean(all_pitches):.1f}')
axes[0].set_xlabel('MIDI Pitch'); axes[0].set_ylabel('Note Count')
axes[0].set_title('Melody Pitch Distribution (all 909 songs)')
axes[0].legend()

pc_counts = Counter(p % 12 for p in all_pitches)
pc_names  = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
axes[1].bar(pc_names, [pc_counts[i] for i in range(12)], color='coral')
axes[1].set_xlabel('Pitch Class'); axes[1].set_ylabel('Count')
axes[1].set_title('Pitch Class Distribution')

plt.tight_layout()
plt.savefig('images/task2_pitch_dist.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.3 Chord Vocabulary

POP909 chord annotations give 176 unique chord types — far richer than Bach's functional harmony.

In [ ]:
top20       = chord_counter.most_common(20)
chords      = [c for c,_ in top20]
counts      = [n for _,n in top20]
total_chords = sum(chord_counter.values())

# group by quality (part after ':')
quality_counter = Counter()
for chord, cnt in chord_counter.items():
    quality = chord.split(':')[1] if ':' in chord else chord
    quality_counter[quality] += cnt
top_qual = quality_counter.most_common(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(chords[::-1], counts[::-1], color='mediumpurple')
axes[0].set_xlabel('Occurrences')
axes[0].set_title(f'Top 20 Chords  (vocab = {len(chord_counter)} unique types)')

axes[1].bar([q for q,_ in top_qual], [c for _,c in top_qual], color='teal')
axes[1].set_xlabel('Quality'); axes[1].set_ylabel('Total Occurrences')
axes[1].set_title('Chord Quality Distribution (top 10)')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig('images/task2_chord_vocab.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 5 chords:")
for chord, cnt in top20[:5]:
    print(f"  {chord:<22} {cnt:>6,}  ({100*cnt/total_chords:.1f}%)")


### 2.4 Song Length & Note Density

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(song_lengths, bins=40, color='goldenrod', edgecolor='white', lw=0.3)
axes[0].axvline(np.mean(song_lengths), color='red', ls='--',
                label=f"Mean: {np.mean(song_lengths):.0f}s")
axes[0].set_xlabel('Duration (s)'); axes[0].set_ylabel('Songs')
axes[0].set_title('Song Length Distribution'); axes[0].legend()

axes[1].hist(note_densities, bins=40, color='seagreen', edgecolor='white', lw=0.3)
axes[1].axvline(np.mean(note_densities), color='red', ls='--',
                label=f"Mean: {np.mean(note_densities):.2f}")
axes[1].set_xlabel('Notes / second'); axes[1].set_ylabel('Songs')
axes[1].set_title('Melody Note Density'); axes[1].legend()

plt.tight_layout()
plt.savefig('images/task2_song_stats.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Data Pipeline

Each song is segmented into **64-slot windows** (≈ 4 bars at 120 BPM, 16th-note resolution) with 50 % overlap.

| Slot value | Meaning |
|-----------|---------|
| 0–127 | MIDI pitch |
| 128 | REST |
| Piano slot | 4 pitches (descending), padded with 0 |

Windows where > 80 % of melody slots are REST are discarded.


In [ ]:
from pop909_dataset import POP909Dataset
import torch

train_ds = POP909Dataset('data/POP909/POP909', split='train')
val_ds   = POP909Dataset('data/POP909/POP909', split='val')

print(f"Training windows:   {len(train_ds):,}")
print(f"Validation windows: {len(val_ds):,}")
print(f"Total:              {len(train_ds)+len(val_ds):,}")

melody, piano = train_ds[0]
print(f"\nSample — melody shape: {tuple(melody.shape)}, piano shape: {tuple(piano.shape)}")
print(f"Melody (first 16 slots): {melody[:16].tolist()}")
print(f"Piano  (first 4 slots):  {piano[:4].tolist()}")
print(f"  128 = REST, 0 = pad")


## 4. Model: Seq2Seq BiLSTM with Bahdanau Attention

```
Melody tokens (64,)
      │
 [Embedding 130→128]
      │
 [BiLSTM ×2 layers, hidden=256]  ← bidirectional: reads melody left-to-right AND right-to-left
      │
 encoder_outputs (64, 512)
      │
  ┌───┴──────────────────────────────┐
  │       BAHDANAU ATTENTION         │
  │  score_j = v · tanh(W1·h_t       │
  │              + W2·encoder_j)     │
  │  α = softmax(scores)             │
  │  context = Σ α_j · encoder_j     │
  └──────────────────────────────────┘
                    │
          [LSTM Decoder ×2 layers]
                    │
           ┌────────┴────────┐
        [Linear ×4]      each head → vocab 129
        voice 0–3         (pitch 0–127, 128=REST)
```


In [ ]:
from harmonizer_model import HarmonizerSeq2Seq
import torch

# Correct signature: embed_dim, hidden_dim, num_layers, dropout
model = HarmonizerSeq2Seq(embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3)

total     = sum(p.numel() for p in model.parameters())
enc_params = sum(p.numel() for p in model.encoder.parameters())
dec_params = sum(p.numel() for p in model.decoder.parameters())

print(f"Total parameters:  {total:,}")
print(f"  Encoder (BiLSTM):{enc_params:,}")
print(f"  Decoder (LSTM):  {dec_params:,}")

# Sanity-check forward pass
model.eval()
with torch.no_grad():
    dummy_melody = torch.randint(0, 128, (2, 64))
    dummy_piano  = torch.zeros(2, 64, 4, dtype=torch.long)
    logits, targets = model(dummy_melody, dummy_piano, teacher_forcing_ratio=0.0)

# logits: (batch*seq, 4*129)  targets: (batch*seq, 4)
print(f"\nForward pass OK")
print(f"  logits shape:  {tuple(logits.shape)}  (batch×seq=128, 4 voices × 129 vocab)")
print(f"  targets shape: {tuple(targets.shape)}")


In [ ]:
# Training curves — available after Colab training
import os, matplotlib.pyplot as plt

losses_path = 'modeling_task2/harmonizer_losses.json'
if os.path.exists(losses_path):
    import json
    L = json.load(open(losses_path))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(L['train'], label='Train')
    ax.plot(L['val'],   label='Val')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('Harmonizer Training Curves')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('images/task2_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    best_epoch = L['val'].index(min(L['val'])) + 1
    print(f"Best val loss: {min(L['val']):.4f}  at epoch {best_epoch}")
else:
    print("No checkpoint yet.  Steps:")
    print("  1. Upload modeling_task2/pop909_cache.pkl to Colab (91 MB)")
    print("  2. Also upload harmonizer_model.py and pop909_dataset.py")
    print("  3. Run colab/task2/train_harmonizer_colab.py  (~30 min T4, ~10 min A100)")
    print("  4. Download harmonizer_best.pt → modeling_task2/harmonizer_best.pt")


## 5. Colab Training

| File to upload | Size |
|----------------|------|
| `modeling_task2/pop909_cache.pkl` | 91 MB |
| `modeling_task2/harmonizer_model.py` | — |
| `modeling_task2/pop909_dataset.py` | — |

Copy cells from `colab/task2/train_harmonizer_colab.py`.
After training, download `harmonizer_best.pt` → place in `modeling_task2/`.

Expected: **T4 ≈ 30 min**, **A100 ≈ 10 min** for 30 epochs.


## 6. Generated Samples

In [ ]:
EVAL_DIR   = Path('evaluation_task2')
CHECKPOINT = Path('modeling_task2/harmonizer_best.pt')

print(f"Checkpoint: {CHECKPOINT}  (exists={CHECKPOINT.exists()})")
print(f"Soundfont:  {SOUNDFONT}   (exists={os.path.exists(SOUNDFONT)})")

if not CHECKPOINT.exists():
    print("\nNo checkpoint yet — train on Colab first, then drop harmonizer_best.pt into modeling_task2/")
else:
    import subprocess
    SONG_IDS = ['001', '042', '100']

    # Generate MIDI files if not already present
    print("\nGenerating harmonizations...")
    for song_id in SONG_IDS:
        mid = EVAL_DIR / f'harmony_{song_id}.mid'
        if mid.exists():
            print(f"  [{song_id}] MIDI already exists, skipping generation")
        else:
            print(f"  [{song_id}] Running generate_harmony.py ...")
            r = subprocess.run(
                [sys.executable, 'modeling_task2/generate_harmony.py', '--song', song_id],
                capture_output=True, text=True)
            if r.returncode == 0:
                print(f"  [{song_id}] Generated → {mid}")
            else:
                print(f"  [{song_id}] ERROR: {r.stderr[-300:]}")

    # Render WAVs and embed players
    print("\nRendering WAV files...")
    html = ''
    for song_id in SONG_IDS:
        for sfx, label in [('_original', 'Original'), ('', 'Generated')]:
            mid = EVAL_DIR / f'harmony_{song_id}{sfx}.mid'
            wav = EVAL_DIR / f'harmony_{song_id}{sfx}.wav'
            print(f"  [{song_id} {label}]  MIDI={mid.exists()}  WAV={wav.exists()}")
            if mid.exists():
                if not wav.exists():
                    pm_obj = pretty_midi.PrettyMIDI(str(mid))
                    midi_to_wav(pm_obj, str(wav))
                if wav.exists():
                    html += audio_tag(str(wav), f'Song {song_id} — {label}')
                else:
                    html += f'<p style="color:red"><b>Song {song_id} {label}</b> — WAV render failed</p>'
            else:
                html += f'<p style="color:orange"><b>Song {song_id} {label}</b> — MIDI not found</p>'

    display(HTML(html))


## 7. Evaluation Metrics

| Metric | Direction | What it measures |
|--------|-----------|-----------------|
| Scale consistency (%) | ↑ | Notes fitting the detected key |
| Voice crossing rate (%) | ↓ | Ordering violations between voices |
| Parallel 5ths rate (%) | ↓ | Forbidden parallel perfect 5ths |
| Pitch KL divergence | ↓ | Distance from real POP909 pitch distribution |


In [ ]:
import json, os
import pandas as pd

metrics_path = 'evaluation_task2/harmony_metrics.json'
if os.path.exists(metrics_path):
    metrics = json.load(open(metrics_path))
    df = pd.DataFrame(metrics).T
    print(df.round(3).to_string())
else:
    print("Run evaluation_task2/evaluate_harmony.py after training to fill this in.")
    placeholder = pd.DataFrame({
        'Our Model':         {'Scale Consistency (%)': '—', 'Voice Crossing (%)': '—',
                              'Parallel 5ths (%)': '—',   'Pitch KL': '—'},
        'Random Baseline':   {'Scale Consistency (%)': 69.9,'Voice Crossing (%)': 96.3,
                              'Parallel 5ths (%)': 0.0,   'Pitch KL': 0.10},
        'Real POP909':       {'Scale Consistency (%)': '~90','Voice Crossing (%)': '~1',
                              'Parallel 5ths (%)': '~0.2','Pitch KL': 0.0},
    }).T
    print(placeholder.to_string())


## 8. Related Work

### DeepBach — Hadjeres et al. (2017)
Bach harmonization via Gibbs sampling over per-voice neural networks.
Iterates until convergence — principled but slow (~seconds per bar).
*vs. ours:* our seq2seq is a single forward pass; less strict on voice-leading but generalises to pop.

### Coconet — Huang et al. (2017)
Dilated CNNs + blocked Gibbs for polyphonic inpainting.
*vs. ours:* we use explicit attention for melody–harmony alignment; Coconet is better for arbitrary masking.

### Music Transformer — Huang et al. (2018)
Relative-attention Transformer for long-form unconditioned piano generation (Maestro).
Closest to Task 1; Task 2 adds explicit melody conditioning.

### Pop Music Transformer — Huang et al. (2020)
Transformer-XL on REMI-tokenised pop MIDI — long-form generation but still unconditioned.
Our work adds the conditioning signal (given melody → generate accompaniment).
